In [13]:
import warnings
warnings.filterwarnings("ignore")

In [14]:
import json
import itertools
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    DBSCAN,
    OPTICS,
    SpectralClustering,
)
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)
from sklearn.neighbors import kneighbors_graph

sns.set(style="whitegrid", context="talk")

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("Warning: umap-learn is not installed. UMAP-based spaces will be skipped.")

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print("Warning: networkx is not installed. Community graph preprocessing may be limited.")

try:
    import igraph as ig
    import leidenalg
    HAS_LEIDEN = True
except ImportError:
    HAS_LEIDEN = False
    print("Warning: igraph/leidenalg not installed. Leiden clustering will be skipped.")

In [15]:
DATA_DIR = Path("../../results/processed_hrv")
INPUT_FILE = DATA_DIR / "basal_v2_clean_with_categories.csv"

OUT_DIR = Path("../../results/clustering_exploration")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TABLE_DIR = OUT_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_DIR = OUT_DIR / "labels"
LABEL_DIR.mkdir(parents=True, exist_ok=True)

META_DIR = OUT_DIR / "metadata"
META_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [16]:
df = pd.read_csv(INPUT_FILE)

print(df.shape)
df.head()

(530, 29)


,sex,age,weight_kg,height_cm_final,height_m_final,imc_final,bp_systolic,bp_diastolic,bp_pam,bp_pp,...,mean_hr_from_rr,height_inconsistent,imc_inconsistent,hr_rr_inconsistent,n_integrity_flags,n_range_flags,n_total_qc_flags,age_group,bmi_cat,bp_cat
0,1,75,70.0,160.0,1.60,27.343750,108,70,82.666667,38,...,80.213904,False,False,False,0,0,0,70-79,normal,normal
1,1,66,52.0,149.0,1.49,23.422368,122,72,88.666667,50,...,62.959077,False,False,False,0,0,0,60-69,normal,normal
2,2,77,81.0,170.0,1.70,28.027682,125,85,98.333333,40,...,68.415051,False,False,False,0,0,0,70-79,overweight,normal
3,1,77,85.0,160.0,1.60,33.203125,126,72,90.000000,54,...,73.260073,False,False,False,0,0,0,70-79,obesity,normal
4,1,73,69.0,151.0,1.51,30.261831,130,80,96.666667,50,...,70.921986,False,False,False,0,0,0,70-79,overweight,normal


In [17]:
hrv_cols = [
    "t2m_pre_mean_rr",
    "t2m_pre_mean_hr",
    "t2m_pre_sdnn",
    "t2m_pre_rmssd",
    "t2m_pre_hf",
    "t2m_pre_lf",
    "t2m_pre_vlf",
]

phys_cont_cols = [
    "age",
    "weight_kg",
    "height_m_final",
    "imc_final",
    "bp_systolic",
    "bp_diastolic",
    "bp_pam",
    "bp_pp",
]

phys_cat_cols = [
    "sex",
    "age_group",
    "bmi_cat",
    "bp_cat",
]

existing_hrv = [c for c in hrv_cols if c in df.columns]
existing_phys_cont = [c for c in phys_cont_cols if c in df.columns]
existing_phys_cat = [c for c in phys_cat_cols if c in df.columns]

existing_hrv, existing_phys_cont

(['t2m_pre_mean_rr',
  't2m_pre_mean_hr',
  't2m_pre_sdnn',
  't2m_pre_rmssd',
  't2m_pre_hf',
  't2m_pre_lf',
  't2m_pre_vlf'],
 ['age',
  'weight_kg',
  'height_m_final',
  'imc_final',
  'bp_systolic',
  'bp_diastolic',
  'bp_pam',
  'bp_pp'])

In [18]:
log_candidates = [
    "t2m_pre_sdnn",
    "t2m_pre_rmssd",
    "t2m_pre_hf",
    "t2m_pre_lf",
    "t2m_pre_vlf",
]

existing_log_candidates = [c for c in log_candidates if c in df.columns]

for col in existing_log_candidates:
    df[f"log_{col}"] = np.where(df[col] > 0, np.log1p(df[col]), np.nan)

log_hrv_cols = [f"log_{c}" for c in existing_log_candidates]
log_hrv_cols

['log_t2m_pre_sdnn',
 'log_t2m_pre_rmssd',
 'log_t2m_pre_hf',
 'log_t2m_pre_lf',
 'log_t2m_pre_vlf']

In [19]:
def impute_and_scale(df_in):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_imp = imputer.fit_transform(df_in)
    X_scaled = scaler.fit_transform(X_imp)

    X_imp_df = pd.DataFrame(X_imp, columns=df_in.columns, index=df_in.index)
    X_scaled_df = pd.DataFrame(X_scaled, columns=df_in.columns, index=df_in.index)

    return X_imp_df, X_scaled_df, imputer, scaler

In [20]:
df_hrv_raw = df[existing_hrv].copy()
df_hrv_log = df[log_hrv_cols].copy()
df_phys = df[existing_phys_cont].copy()
df_integrated = df[log_hrv_cols + existing_phys_cont].copy()

X_hrv_raw_imp, X_hrv_raw_scaled, _, _ = impute_and_scale(df_hrv_raw)
X_hrv_log_imp, X_hrv_log_scaled, _, _ = impute_and_scale(df_hrv_log)
X_phys_imp, X_phys_scaled, _, _ = impute_and_scale(df_phys)
X_integrated_imp, X_integrated_scaled, _, _ = impute_and_scale(df_integrated)

print(X_hrv_raw_scaled.shape)
print(X_hrv_log_scaled.shape)
print(X_phys_scaled.shape)
print(X_integrated_scaled.shape)

(530, 7)
(530, 5)
(530, 8)
(530, 13)


In [21]:
def build_pca_space(X_df, n_components):
    pca = PCA(n_components=n_components, random_state=42) if "random_state" in PCA.__init__.__code__.co_varnames else PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_df)
    cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
    return pd.DataFrame(X_pca, columns=cols, index=X_df.index), pca

In [22]:
pca_spaces = {}

for name, X in {
    "hrv_log": X_hrv_log_scaled,
    "integrated": X_integrated_scaled,
}.items():
    for n_comp in [2, 3, 4, 5]:
        if X.shape[1] >= n_comp:
            X_pca_df, pca_model = build_pca_space(X, n_comp)
            pca_spaces[f"pca_{name}_{n_comp}d"] = X_pca_df

list(pca_spaces.keys())

['pca_hrv_log_2d',
 'pca_hrv_log_3d',
 'pca_hrv_log_4d',
 'pca_hrv_log_5d',
 'pca_integrated_2d',
 'pca_integrated_3d',
 'pca_integrated_4d',
 'pca_integrated_5d']

In [23]:
umap_spaces = {}

if HAS_UMAP:
    for name, X in {
        "hrv_log": X_hrv_log_scaled,
        "integrated": X_integrated_scaled,
    }.items():
        for n_neighbors in [10, 15, 30]:
            for min_dist in [0.0, 0.1, 0.3]:
                reducer = umap.UMAP(
                    n_neighbors=n_neighbors,
                    min_dist=min_dist,
                    n_components=2,
                    random_state=42
                )
                X_umap = reducer.fit_transform(X)
                key = f"umap_{name}_nn{n_neighbors}_md{str(min_dist).replace('.', '')}"
                umap_spaces[key] = pd.DataFrame(X_umap, columns=["UMAP1", "UMAP2"], index=X.index)

list(umap_spaces.keys())[:10]

['umap_hrv_log_nn10_md00',
 'umap_hrv_log_nn10_md01',
 'umap_hrv_log_nn10_md03',
 'umap_hrv_log_nn15_md00',
 'umap_hrv_log_nn15_md01',
 'umap_hrv_log_nn15_md03',
 'umap_hrv_log_nn30_md00',
 'umap_hrv_log_nn30_md01',
 'umap_hrv_log_nn30_md03',
 'umap_integrated_nn10_md00']

In [24]:
tsne_spaces = {}

for name, X in {
    "hrv_log": X_hrv_log_scaled,
    "integrated": X_integrated_scaled,
}.items():
    for perplexity in [10, 20, 30, 40]:
        tsne = TSNE(
            n_components=2,
            perplexity=perplexity,
            learning_rate="auto",
            init="pca",
            random_state=42
        )
        X_tsne = tsne.fit_transform(X)
        key = f"tsne_{name}_perp{perplexity}"
        tsne_spaces[key] = pd.DataFrame(X_tsne, columns=["TSNE1", "TSNE2"], index=X.index)

list(tsne_spaces.keys())

['tsne_hrv_log_perp10',
 'tsne_hrv_log_perp20',
 'tsne_hrv_log_perp30',
 'tsne_hrv_log_perp40',
 'tsne_integrated_perp10',
 'tsne_integrated_perp20',
 'tsne_integrated_perp30',
 'tsne_integrated_perp40']

In [25]:
input_spaces = {
    "hrv_raw_scaled": X_hrv_raw_scaled,
    "hrv_log_scaled": X_hrv_log_scaled,
    "phys_scaled": X_phys_scaled,
    "integrated_scaled": X_integrated_scaled,
}

input_spaces.update(pca_spaces)
input_spaces.update(umap_spaces)
input_spaces.update(tsne_spaces)

len(input_spaces), list(input_spaces.keys())[:20]

(38,
 ['hrv_raw_scaled',
  'hrv_log_scaled',
  'phys_scaled',
  'integrated_scaled',
  'pca_hrv_log_2d',
  'pca_hrv_log_3d',
  'pca_hrv_log_4d',
  'pca_hrv_log_5d',
  'pca_integrated_2d',
  'pca_integrated_3d',
  'pca_integrated_4d',
  'pca_integrated_5d',
  'umap_hrv_log_nn10_md00',
  'umap_hrv_log_nn10_md01',
  'umap_hrv_log_nn10_md03',
  'umap_hrv_log_nn15_md00',
  'umap_hrv_log_nn15_md01',
  'umap_hrv_log_nn15_md03',
  'umap_hrv_log_nn30_md00',
  'umap_hrv_log_nn30_md01'])

In [26]:
def get_cluster_counts(labels):
    labels = np.asarray(labels)
    valid = labels[labels != -1]
    counts = Counter(valid)
    return counts

def cluster_size_entropy(labels):
    counts = get_cluster_counts(labels)
    if len(counts) == 0:
        return np.nan
    probs = np.array(list(counts.values()), dtype=float)
    probs = probs / probs.sum()
    return -(probs * np.log2(probs)).sum()

def compute_basic_cluster_stats(labels):
    labels = np.asarray(labels)
    counts = get_cluster_counts(labels)

    n_total = len(labels)
    n_noise = int(np.sum(labels == -1))
    n_clustered = n_total - n_noise
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    if len(counts) > 0:
        sizes = np.array(list(counts.values()))
        min_size = int(sizes.min())
        max_size = int(sizes.max())
        median_size = float(np.median(sizes))
        imbalance_ratio = float(max_size / min_size) if min_size > 0 else np.nan
        entropy = cluster_size_entropy(labels)
    else:
        min_size = np.nan
        max_size = np.nan
        median_size = np.nan
        imbalance_ratio = np.nan
        entropy = np.nan

    return {
        "n_clusters": n_clusters,
        "n_noise": n_noise,
        "noise_fraction": n_noise / n_total,
        "n_clustered": n_clustered,
        "min_cluster_size": min_size,
        "max_cluster_size": max_size,
        "median_cluster_size": median_size,
        "cluster_size_entropy": entropy,
        "imbalance_ratio": imbalance_ratio,
    }

def compute_internal_metrics(X, labels):
    labels = np.asarray(labels)
    unique_valid = set(labels) - {-1}

    if len(unique_valid) < 2:
        return {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
        }

    mask = labels != -1
    X_eval = X[mask]
    labels_eval = labels[mask]

    if len(np.unique(labels_eval)) < 2:
        return {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
        }

    return {
        "silhouette": silhouette_score(X_eval, labels_eval),
        "calinski_harabasz": calinski_harabasz_score(X_eval, labels_eval),
        "davies_bouldin": davies_bouldin_score(X_eval, labels_eval),
    }

In [27]:
def run_kmeans(X, n_clusters, random_state=42):
    model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=20)
    labels = model.fit_predict(X)
    return labels

def run_gmm(X, n_components, covariance_type, random_state=42):
    model = GaussianMixture(
        n_components=n_components,
        covariance_type=covariance_type,
        random_state=random_state
    )
    labels = model.fit_predict(X)
    return labels

def run_agglomerative(X, n_clusters, linkage, metric="euclidean"):
    if linkage == "ward":
        model = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
    else:
        model = AgglomerativeClustering(
            n_clusters=n_clusters,
            linkage=linkage,
            metric=metric
        )
    labels = model.fit_predict(X)
    return labels

def run_dbscan(X, eps, min_samples, metric="euclidean"):
    model = DBSCAN(eps=eps, min_samples=min_samples, metric=metric)
    labels = model.fit_predict(X)
    return labels

def run_optics(X, min_samples, xi, min_cluster_size):
    model = OPTICS(
        min_samples=min_samples,
        xi=xi,
        min_cluster_size=min_cluster_size
    )
    labels = model.fit_predict(X)
    return labels

def run_spectral(X, n_clusters, affinity="nearest_neighbors", random_state=42):
    model = SpectralClustering(
        n_clusters=n_clusters,
        affinity=affinity,
        random_state=random_state,
        assign_labels="kmeans"
    )
    labels = model.fit_predict(X)
    return labels

In [28]:
def run_leiden_knn(X, n_neighbors=15, resolution=1.0):
    if not HAS_LEIDEN:
        raise ImportError("Leiden dependencies not available.")

    knn = kneighbors_graph(X, n_neighbors=n_neighbors, mode="connectivity", include_self=False)
    sources, targets = knn.nonzero()
    edges = list(zip(sources.tolist(), targets.tolist()))

    g = ig.Graph(n=X.shape[0], edges=edges, directed=False)
    partition = leidenalg.find_partition(
        g,
        leidenalg.RBConfigurationVertexPartition,
        resolution_parameter=resolution
    )
    labels = np.array(partition.membership)
    return labels

In [29]:
k_values = [2, 3, 4, 5, 6, 7, 8, 9, 10]
gmm_covariances = ["full", "diag", "tied", "spherical"]
agg_linkages = ["ward", "complete", "average", "single"]
dbscan_eps_values = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]
dbscan_min_samples_values = [3, 5, 10, 15]
optics_min_samples_values = [3, 5, 10, 15]
optics_xi_values = [0.01, 0.03, 0.05, 0.1]
optics_min_cluster_sizes = [0.03, 0.05, 0.1]
spectral_affinities = ["nearest_neighbors", "rbf"]
leiden_n_neighbors = [10, 15, 30]
leiden_resolutions = [0.5, 1.0, 1.5, 2.0]

In [30]:
experiments = []

for space_name, X_df in input_spaces.items():
    X = X_df.values

    # KMeans
    for k in k_values:
        experiments.append({
            "algorithm": "kmeans",
            "space_name": space_name,
            "params": {"n_clusters": k},
        })

    # GMM
    for k in k_values:
        for cov in gmm_covariances:
            experiments.append({
                "algorithm": "gmm",
                "space_name": space_name,
                "params": {"n_components": k, "covariance_type": cov},
            })

    # Agglomerative
    for k in k_values:
        for linkage in agg_linkages:
            experiments.append({
                "algorithm": "agglomerative",
                "space_name": space_name,
                "params": {"n_clusters": k, "linkage": linkage},
            })

    # DBSCAN
    for eps in dbscan_eps_values:
        for min_samples in dbscan_min_samples_values:
            experiments.append({
                "algorithm": "dbscan",
                "space_name": space_name,
                "params": {"eps": eps, "min_samples": min_samples},
            })

    # OPTICS
    for min_samples in optics_min_samples_values:
        for xi in optics_xi_values:
            for min_cluster_size in optics_min_cluster_sizes:
                experiments.append({
                    "algorithm": "optics",
                    "space_name": space_name,
                    "params": {
                        "min_samples": min_samples,
                        "xi": xi,
                        "min_cluster_size": min_cluster_size,
                    },
                })

    # Spectral
    for k in k_values:
        for affinity in spectral_affinities:
            experiments.append({
                "algorithm": "spectral",
                "space_name": space_name,
                "params": {"n_clusters": k, "affinity": affinity},
            })

    # Leiden
    if HAS_LEIDEN:
        for nn in leiden_n_neighbors:
            for res in leiden_resolutions:
                experiments.append({
                    "algorithm": "leiden_knn",
                    "space_name": space_name,
                    "params": {"n_neighbors": nn, "resolution": res},
                })

len(experiments)

6954

In [31]:
def run_single_experiment(exp, input_spaces, random_state=42):
    algorithm = exp["algorithm"]
    space_name = exp["space_name"]
    params = exp["params"]

    X_df = input_spaces[space_name]
    X = X_df.values

    try:
        if algorithm == "kmeans":
            labels = run_kmeans(X, n_clusters=params["n_clusters"], random_state=random_state)

        elif algorithm == "gmm":
            labels = run_gmm(
                X,
                n_components=params["n_components"],
                covariance_type=params["covariance_type"],
                random_state=random_state
            )

        elif algorithm == "agglomerative":
            labels = run_agglomerative(
                X,
                n_clusters=params["n_clusters"],
                linkage=params["linkage"]
            )

        elif algorithm == "dbscan":
            labels = run_dbscan(
                X,
                eps=params["eps"],
                min_samples=params["min_samples"]
            )

        elif algorithm == "optics":
            labels = run_optics(
                X,
                min_samples=params["min_samples"],
                xi=params["xi"],
                min_cluster_size=params["min_cluster_size"]
            )

        elif algorithm == "spectral":
            labels = run_spectral(
                X,
                n_clusters=params["n_clusters"],
                affinity=params["affinity"],
                random_state=random_state
            )

        elif algorithm == "leiden_knn":
            labels = run_leiden_knn(
                X,
                n_neighbors=params["n_neighbors"],
                resolution=params["resolution"]
            )

        else:
            raise ValueError(f"Unknown algorithm: {algorithm}")

        basic_stats = compute_basic_cluster_stats(labels)
        internal_metrics = compute_internal_metrics(X, labels)

        result = {
            "status": "ok",
            "algorithm": algorithm,
            "space_name": space_name,
            "params_json": json.dumps(params, sort_keys=True),
            **basic_stats,
            **internal_metrics,
        }

        return result, labels

    except Exception as e:
        result = {
            "status": "error",
            "algorithm": algorithm,
            "space_name": space_name,
            "params_json": json.dumps(params, sort_keys=True),
            "error_message": str(e),
        }
        return result, None

In [32]:
results = []
label_index = []

for i, exp in enumerate(experiments):
    result, labels = run_single_experiment(exp, input_spaces=input_spaces, random_state=42)

    result["experiment_id"] = i
    results.append(result)

    if labels is not None:
        label_file = LABEL_DIR / f"labels_exp_{i:05d}.csv"
        pd.DataFrame({
            "row_id": df.index,
            "cluster_label": labels
        }).to_csv(label_file, index=False)

        label_index.append({
            "experiment_id": i,
            "label_file": str(label_file),
            "algorithm": exp["algorithm"],
            "space_name": exp["space_name"],
            "params_json": json.dumps(exp["params"], sort_keys=True),
        })

results_df = pd.DataFrame(results)
label_index_df = pd.DataFrame(label_index)

results_df.head()

,status,algorithm,space_name,params_json,n_clusters,n_noise,noise_fraction,n_clustered,min_cluster_size,max_cluster_size,median_cluster_size,cluster_size_entropy,imbalance_ratio,silhouette,calinski_harabasz,davies_bouldin,experiment_id
0,ok,kmeans,hrv_raw_scaled,"{""n_clusters"": 2}",2,0,0.0,530,37.0,493.0,265.0,0.365219,13.324324,0.598903,170.712611,1.192077,0
1,ok,kmeans,hrv_raw_scaled,"{""n_clusters"": 3}",3,0,0.0,530,35.0,310.0,185.0,1.241492,8.857143,0.273415,174.708670,1.232019,1
2,ok,kmeans,hrv_raw_scaled,"{""n_clusters"": 4}",4,0,0.0,530,29.0,175.0,163.0,1.802967,6.034483,0.242396,156.408032,1.252247,2
3,ok,kmeans,hrv_raw_scaled,"{""n_clusters"": 5}",5,0,0.0,530,15.0,172.0,157.0,1.889536,11.466667,0.257802,153.493475,1.274984,3
4,ok,kmeans,hrv_raw_scaled,"{""n_clusters"": 6}",6,0,0.0,530,10.0,172.0,84.5,1.991712,17.200000,0.277803,152.634911,1.167301,4
